<a href="https://colab.research.google.com/github/edwardsnj/pride-study-retrieval-cofest-2026/blob/main/notebooks/TextEmbedding%2BTFIDF_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#
# Import necessary modules
#

import sys, os, httpimport

with httpimport.github_repo("EdwardsLabProjects", "pride-study-retrieval-cofest-2026", ref="main"):
    from notebooks.lib import *

set_random_seed(state=None)

Version: 1.0.34
Using random seed: 7091794


In [2]:
#
# Get embeddings and known true positives
#

md,emb = embeddings("openai-3-small")
embdim = emb.shape[0]
tp,tn = knownstudies()

allacc = set(emb.columns)
tp &= allacc
tn &= allacc

assert len(tp & tn) == 0, "TP and TN should not intersect!"

# Print out some details...
print(f"Embeddings: {emb.shape}")
print(f"True Pos: {len(tp)}")
print(f"True Neg: {len(tn)}")

# allacc,avgemb = select_by_embedding_proximity(tp,emb,1000)
# print(f"All Acc: {len(allacc)}")

Embeddings: (1536, 36696)
True Pos: 86
True Neg: 47


In [3]:
#
# Create train/test split
#
TESTFRAC = 0.2
BACKGROUND_OVERSAMPLE = 25

train_acc, train_y, test_acc, test_y = split_train_test(allacc, tp, tn,
    test_size=TESTFRAC, bgsize=BACKGROUND_OVERSAMPLE)

print(f"\nTrain set size: {len(train_acc)}")
print(f"  True Positives in Train: {train_y.count(1)}")
print(f"  True Negatives in Train: {train_y.count(0)}")

print(f"\nTest set size: {len(test_acc)}")
print(f"  True Positives in Test: {test_y.count(1)}")
print(f"  True Negatives in Test: {test_y.count(0)}")


Train set size: 1788
  True Positives in Train: 68
  True Negatives in Train: 1720

Test set size: 448
  True Positives in Test: 18
  True Negatives in Test: 430


In [4]:
# Build the TF-IDF features from train data only,
# but build feature vectors for all documents

from sklearn.feature_extraction import text
stop_words = list(set(list(text.ENGLISH_STOP_WORDS) + """

""".split()))

tfidf_extra_args = dict(positive_only=False, # Use only positive set for word selection
                        min_df=1, # minimum documents word is in
                        max_df=1.0, # maximum documents word is in
                        max_features=None, # max number of words to use
                        stop_words=stop_words) # words to ignore

tfidf,tfidf_model = create_tfidf_features(md, train_acc, train_y,
                                          test_acc, **tfidf_extra_args)

print(f"TF-IDF features shape: {tfidf.shape}")

TF-IDF features shape: (35422, 2236)


In [5]:
logreg_extra_args = dict(class_weight='balanced',C=10.0,
                         penalty='l1', solver='liblinear',
                         use_embed=True, use_tfidf=True)
trained_model = train_document_classifier(emb, tfidf,
                                          train_acc, train_y,
                                          test_acc, test_y,
                                          **logreg_extra_args)
print("NZ Coeffs:",sum(*[1*(xi!=0) for xi in trained_model.coef_]))

Training data shape: (1788, 36958) (Positives: 68, Negatives: 1720)
Testing data shape: (448, 36958) (Positives: 18, Negatives: 430)

--- Model Evaluation ---
Accuracy: 0.9933

Classification Report:
                precision    recall  f1-score   support

Background (0)       1.00      0.99      1.00       430
 Seed-like (1)       0.86      1.00      0.92        18

      accuracy                           0.99       448
     macro avg       0.93      1.00      0.96       448
  weighted avg       0.99      0.99      0.99       448

NZ Coeffs: 44


In [6]:
nzemb,nztfidf,topfeat = top_features(trained_model,tfidf_model,nembed=embdim,**logreg_extra_args)
print(f"Number of sem. embedding dimensions with non-zero coefficients: {nzemb}")
print(f"Number of TF-IDF dimensions with non-zero coefficients: {nztfidf}")
print("\nTop Features:")
display(topfeat)

show_top_feature_examples(topfeat, md)

Number of sem. embedding dimensions with non-zero coefficients: 1
Number of TF-IDF dimensions with non-zero coefficients: 43

Top Features:


,Feature,Coefficient
27110,pngase,129.216335
12516,deamidation,38.708530
34018,tof,-33.514452
17071,glycosite,22.172386
17078,glycosylation,17.695359
12672,deglycosylated,16.360529
7748,asn,14.317433
17077,glycosylated,13.482632
9356,bx,13.425159
17887,hepatobiliary,-13.311078



=== 'pngase' (coef: +129.2163) ===
  PXD001432: The beads were washed with 40 µl 1 x G7 buffer (NEB, Frankfurt a.M., Germany) and then incubated with 20 µl 1x G7 buffer containing 500 U glyerol-free PNGase F (NEB) at 37 °C in a thermomixer for 6 h to release the glycopeptides from the beads.
  PXD010911: 1 μL (500 units) of PNGase F was added to one tube of peptides in each pair of aliquots; the second tube was not treated with the glycosidase and served as the control.
  PXD007267: The EndoH released glycans were removed by centrifugation and the remaining material was exchanged into 40 mM ammonium bicarbonate buffer in H2[18O] (Sigma), and subsequently digested with 100 units of N-glycosidase F (PNGaseF, Roche) dissolved in H2[18O].
  PXD007268: Aliquots of the eluted Ricin‐bound and ConA‐bound fractions were independently subjected to EndoH followed by PNGaseF digestion (Roche Applied Science), subjected to SDS–PAGE and the gel lanes were then divided into 10 independent slices.
  

In [7]:
results = score_all_studies(trained_model, emb, md, tfidf_model, train_acc, tp, tn)
display(results)

Scoring studies: 100%|##########| 74/74 [00:40<00:00,  1.81it/s]


,prideacc,title,probability,in_training,true_positive,true_negative
0,PXD031068,Secretome analysis of basidiomycetous yeast Ta...,1.0,False,False,False
1,PXD029218,Site-specific glycosylation patterns of SARS-C...,1.0,False,False,False
2,PXD060969,Identification and validation of protective gl...,1.0,False,False,False
3,PXD004361,HILIC and ERLIC Enrichment of Glycopeptides De...,1.0,True,True,False
4,PXD012589,Sweet and sour Ehrlichia: glycoproteomics and ...,1.0,False,True,False
...,...,...,...,...,...,...
36691,PXD029545,Quantitative phosphoproteomic analysis of Eime...,0.0,False,False,False
36692,PXD021653,Brain proteome analysis in Alzheimer's disease,0.0,False,False,False
36693,PXD017972,Transgenerational long-term effects of the emb...,0.0,False,False,False
36694,PXD041218,Progresses and challenges of Strongyloides spp...,0.0,False,False,False
